In [1]:
import hashlib
import os
import platform
import uuid
import sys

STUDENT_ID = 20244081126

CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV")
print(f"[Conda 环境] {CONDA_ENV}")

MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号] {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID] {RUN_ID}")
print(f"[Python] {sys.version.split()[0]}")

[Conda 环境] python_course
[学号] 20244081126
[机器指纹] 81f3762c7e6b4435
[RUN_ID] 20244081126-81f3762c7e6b4435
[Python] 3.11.17


# 实验1：正则表达式的基础匹配

- `\d+` 匹配一个或多个数字
- `\s+` 匹配一个或多个空白字符
- `\bcat\b` 中的 `\b` 是单词边界，只匹配独立的 "cat"，不会匹配 "cats" 中的 "cat"
- 交替 `|` 从左到右优先匹配
- 量词：`*` 0次或多次，`+` 至少1次，`?` 0或1次，`{2}` 恰好2次，`{1,2}` 1到2次

In [2]:
import re

text1 = "cat on mat; later: cats and dogs"
print("数字 \\d+ :", re.findall(r"\d+", text1))
print("空白 \\s+ :", re.findall(r"\s+", text1))
print("单词 \\bcat\\b :", re.findall(r"\bcat\b", text1))

数字 \d+ : []
空白 \s+ : [' ', ' ', ' ', ' ', ' ', ' ']
单词 \bcat\b : ['cat']


In [3]:
text2 = "cat dog bird CAT"
print("cat|dog :", re.findall(r"cat|dog", text2))
print("a|aa|aaa :", re.findall(r"a|aa|aaa", "aaaa"))
print("aaa|aa|a :", re.findall(r"aaa|aa|a", "aaaa"))

cat|dog : ['cat', 'dog']
a|aa|aaa : ['a', 'a', 'a', 'a']
aaa|aa|a : ['aaa', 'a']


In [4]:
text3 = "ct cat caat caaat"
for pat in [r"ca*t", r"ca+t", r"ca?t", r"ca{2}t", r"ca{1,2}t"]:
    print(f"{pat:10s} -> {re.findall(pat, text3)}")

ca*t       -> ['ct', 'cat', 'caat', 'caaat']
ca+t       -> ['cat', 'caat', 'caaat']
ca?t       -> ['ct', 'cat']
ca{2}t     -> ['caat']
ca{1,2}t   -> ['cat', 'caat']


# 实验2：分组、回溯引用与替换

- 捕获组用 `(...)`，`re.findall` 返回元组列表
- 后向引用 `\1` 引用第一个捕获组，可匹配连续重复单词
- `re.sub(pattern, repl, text)` 用捕获组替换，替换串中用 `\1`、`\2` 引用

In [5]:
text4 = "联系: alice@example.com; 备选: bob@test.org"
emails = re.findall(r"([\w.+-]+)@([\w.-]+)", text4)
for user, domain in emails:
    print(f"用户名: {user}, 域名: {domain}")

用户名: alice, 域名: example.com
用户名: bob, 域名: test.org


In [6]:
text5 = "cat cat, dog cat, bird bird"
print(re.findall(r"\b(\w+)\s+\1\b", text5))

['cat', 'bird']


In [7]:
text6 = "2024-05-01\n2024/05/02\n2024.05.03"
print("转换前:")
print(text6)
converted = re.sub(r"(\d{4})[-/.](\d{2})[-/.](\d{2})", r"\2-\3-\1", text6)
print("转换后:")
print(converted)

转换前:
2024-05-01
2024/05/02
2024.05.03
转换后:
05-01-2024
05-02-2024
05-03-2024


In [8]:
text7 = "联系我: 123-456-7890 或 987.654.3210"
converted_phone = re.sub(r"(\d{3})[-.](\d{3})[-.](\d{4})", r"(\1) \2-\3", text7)
print(converted_phone)

联系我: (123) 456-7890 或 (987) 654-3210


# 实验3：BeautifulSoup 解析 HTML

- `BeautifulSoup(html_doc, "html.parser")` 构建解析树
- `find_all(tag, class_=...)` 按标签和类名查找
- `get_text()` 提取节点可见文本

In [9]:
from bs4 import BeautifulSoup

html_doc = """<html><head><title>实验课程</title></head><body>
<h1>数据分析</h1>
<p class="course">第六讲: 正则表达式</p>
<p class="course">第七讲: 结构化数据</p>
<a href="https://example.org/lecture06">讲义六</a>
<a href="https://example.org/lecture07">讲义七</a>
</body></html>"""

soup = BeautifulSoup(html_doc, "html.parser")

In [10]:
print("页面标题文本:", soup.title.get_text())
print("h1 标签名:", soup.h1.name)
print("h1 文本:", soup.h1.get_text())

页面标题文本: 实验课程
h1 标签名: h1
h1 文本: 数据分析


In [11]:
courses = soup.find_all("p", class_="course")
print("class=course 的段落:")
for p in courses:
    print("  ", p.get_text())

links = soup.find_all("a")
print("所有链接:")
for a in links:
    print(f"  文字: {a.get_text()} | href: {a.get('href')}")

class=course 的段落:
   第六讲: 正则表达式
   第七讲: 结构化数据
所有链接:
  文字: 讲义六 | href: https://example.org/lecture06
  文字: 讲义七 | href: https://example.org/lecture07


In [12]:
print("=== 整个文档 get_text() ===")
print(soup.get_text())
print("=== 单个段落 get_text() ===")
print(soup.find("p", class_="course").get_text())
print("=== 单个链接 get_text() ===")
print(soup.find("a").get_text())

=== 整个文档 get_text() ===
实验课程
数据分析
第六讲: 正则表达式
第七讲: 结构化数据
讲义六
讲义七

=== 单个段落 get_text() ===
第六讲: 正则表达式
=== 单个链接 get_text() ===
讲义六
